In [3]:
# GLOBAL CONST 
from pathlib import Path

MODULE_DIR = Path.cwd().resolve().parent
RAW_DATA_DIR = MODULE_DIR / "data"/ "raw"

EXTERNAL_TOOLS_DIR = MODULE_DIR / "external_tools"

In [4]:
from __future__ import annotations
import time
from dataclasses import dataclass
from datetime import datetime, timezone
import pandas as pd
import requests

@dataclass(slots=True)
class USGSQueryConfig:
	min_latitude: float = 26.0
	max_latitude: float = 31.0
	min_longitude: float = 80.0
	max_longitude: float = 89.0
	min_magnitude: float = 2.5
	start_year: int = 1970
	end_year: int | None = None  # None defaults to right now
	url: str = "https://earthquake.usgs.gov/fdsnws/event/1/query"


def fetch_earthquake_data(
	config: USGSQueryConfig,
	chunk_years: int = 1,
	request_delay: float = 0.1,
) -> pd.DataFrame:
	"""Fetches earthquake data from USGS API in yearly chunks to bypass record limits.

	:param config: USGSQueryConfig instance with spatial & temporal bounds.
	:param chunk_years: Number of years to fetch per API request (default 1).
	:param request_delay: Sleep duration (in seconds) between API requests.
	:return: Pandas DataFrame containing earthquake records.
	"""
	now_utc = datetime.now(timezone.utc)
	target_end_year = (
		config.end_year if config.end_year is not None else now_utc.year
	)

	all_records = []

	print(
		f"Fetching earthquake data from {config.start_year} to {target_end_year} "
		f"(Chunk size: {chunk_years} year{'s' if chunk_years > 1 else ''})..."
	)

	for current_start in range(
		config.start_year, target_end_year + 1, chunk_years
	):
		current_end = min(current_start + chunk_years - 1, target_end_year)

		start_str = f"{current_start}-01-01T00:00:00"

		# If reaching the current year with no fixed historical end_year, fetch up to right now
		if current_end == now_utc.year and config.end_year is None:
			end_str = now_utc.strftime("%Y-%m-%dT%H:%M:%S")
		else:
			end_str = f"{current_end}-12-31T23:59:59"

		params = {
			"format": "geojson",
			"minlatitude": config.min_latitude,
			"maxlatitude": config.max_latitude,
			"minlongitude": config.min_longitude,
			"maxlongitude": config.max_longitude,
			"minmagnitude": config.min_magnitude,
			"starttime": start_str,
			"endtime": end_str,
			"limit": 20000,
			"orderby": "time-asc",
		}

		try:
			response = requests.get(config.url, params=params, timeout=30)
			response.raise_for_status()
			data = response.json()
			features = data.get("features", [])

			print(
				f"[{current_start} - {current_end}] Downloaded {len(features)} events"
			)

			for feature in features:
				props = feature["properties"]
				coords = feature["geometry"][
					"coordinates"
				]  # [longitude, latitude, depth]

				all_records.append({
					"id" : feature.get("id"),
					"time": pd.to_datetime(props["time"], unit="ms", utc=True),
					"latitude": coords[1],
					"longitude": coords[0],
					"depth_km": coords[2],
					"mag": props.get("mag"),
					"magType": props.get("magType"),
					"place": props.get("place"),
					"nst": props.get("nst"),
					"gap": props.get("gap"),
					"dmin": props.get("dmin"),
					"rms": props.get("rms"),
					"horizontalError": props.get("horizontalError"),
					"depthError": props.get("depthError"),
					"magError": props.get("magError"),
					"felt": props.get("felt"),
					"cdi": props.get("cdi"),
					"mmi": props.get("mmi"),
					"sig": props.get("sig"),
				})

		except requests.exceptions.RequestException as e:
			print(
				f"Failed to fetch data for range {current_start}-{current_end}: {e}"
			)

		time.sleep(request_delay)

	if not all_records:
		return pd.DataFrame()

	df = pd.DataFrame(all_records)
	df = df.drop_duplicates(subset=["time", "latitude", "longitude"]).reset_index(
		drop=True
	)

	print(f"\nTotal records retrieved: {len(df)}")
	if not df.empty:
		print(f"Latest earthquake recorded: {df['time'].max()}")

	return df


query_config = USGSQueryConfig(
	url="https://earthquake.usgs.gov/fdsnws/event/1/query",
	min_latitude=26.0,
	max_latitude=31.0,
	min_longitude=80.0,
	max_longitude=89.0,
	min_magnitude=2.5,
	start_year=1970,
	end_year=None,  # Fetches up to the current second
)
# Fetch in 20-year chunks
df_earthquakes = fetch_earthquake_data(query_config, chunk_years=20, request_delay=0.1)
df_earthquakes.to_csv(RAW_DATA_DIR / "earthquake_data.csv", index=False)
display(df_earthquakes)

Fetching earthquake data from 1970 to 2026 (Chunk size: 20 years)...
[1970 - 1989] Downloaded 168 events
[1990 - 2009] Downloaded 723 events
[2010 - 2026] Downloaded 833 events

Total records retrieved: 1724
Latest earthquake recorded: 2026-09-15 22:43:11.521000+00:00


,id,time,latitude,longitude,depth_km,mag,magType,place,nst,gap,dmin,rms,horizontalError,depthError,magError,felt,cdi,mmi,sig
0,iscgem799042,1970-02-12 01:51:48.800000+00:00,29.2760,81.6480,15.2,5.40,mw,"48 km N of Dailekh, Nepal",NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,449
1,iscgem799449,1970-02-26 19:30:06.160000+00:00,27.6530,85.9140,10.0,5.41,mw,"34 km S of Kod?ri??, Nepal",NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,450
2,iscgem793928,1970-07-21 15:37:43.570000+00:00,27.9770,84.7600,15.0,5.24,mw,"46 km NE of Bharatpur, Nepal",NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,422
3,iscgem784033,1971-05-03 00:33:25.850000+00:00,30.7910,84.3820,19.7,5.50,mw,"181 km NNW of Saga, China",NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,6.836,465
4,iscgem775960,1972-02-04 14:08:22.930000+00:00,30.3380,84.4540,15.0,5.28,mw,"134 km NW of Saga, China",NaN,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1719,us7000tbwb,2026-08-26 02:52:10+00:00,28.2710,85.5150,0.0,5.20,ms_vx,"55 km NW of Kodāri̇̄, Nepal",0.0,NaN,NaN,3.90,None,None,None,2.0,2.0,NaN,601
1720,us7000tc90,2026-08-26 06:00:35+00:00,28.3281,85.4798,0.0,4.20,ms_vx,"62 km NW of Kodāri̇̄, Nepal",0.0,NaN,NaN,1.30,None,None,None,NaN,NaN,NaN,271
1721,us7000tfur,2026-09-08 16:08:34.802000+00:00,29.2121,83.9688,10.0,4.90,mb,"91 km NNE of Chitre, Nepal",104.0,53.0,10.153,0.77,None,None,None,14.0,4.5,NaN,376
1722,us7000tiyx,2026-09-15 20:06:11.362000+00:00,29.4419,81.6398,10.0,4.10,mb,"55 km WNW of Jumla, Nepal",18.0,175.0,8.283,0.87,None,None,None,NaN,NaN,NaN,259


In [5]:
from __future__ import annotations

import sys
import urllib.request
import zipfile
from dataclasses import dataclass
from pathlib import Path


@dataclass(slots=True)
class DatasetConfig:
	name: str
	url: str
	download_path: Path
	check_file: Path
	extract_dir: Path | None = None
	extract_zip: bool = False


def download_with_retries(
	url: str,
	dest: Path,
	max_retries: int = 3,
):
	headers = {
		"User-Agent": (
			"Mozilla/5.0 (X11; Linux x86_64) "
			"AppleWebKit/537.36 (KHTML, like Gecko) "
			"Chrome/120.0.0.0 Safari/537.36"
		)
	}

	for attempt in range(1, max_retries + 1):
		try:
			print(f"Connecting (Attempt {attempt}/{max_retries})...")

			req = urllib.request.Request(url, headers=headers)

			with urllib.request.urlopen(req, timeout=60) as response:
				# Edge Case 1: Create download parent directory ONLY right before writing
				dest.parent.mkdir(parents=True, exist_ok=True)

				with open(dest, "wb") as out_file:
					total_size = int(response.headers.get("Content-Length", 0))
					downloaded = 0
					block_size = 128 * 1024

					while True:
						buffer = response.read(block_size)

						if not buffer:
							break

						downloaded += len(buffer)
						out_file.write(buffer)

						if total_size > 0:
							percent = min(
								100,
								int(downloaded * 100 / total_size),
							)

							sys.stdout.write(
								f"\rDownloading: {percent}% "
								f"({downloaded / 1024**2:.1f}/"
								f"{total_size / 1024**2:.1f} MB)"
							)
							sys.stdout.flush()

					print()

			if total_size > 0 and downloaded < total_size:
				raise IOError(
					f"Incomplete download ({downloaded}/{total_size} bytes)"
				)

			return

		except (Exception, KeyboardInterrupt) as e:
			print(f"\nAttempt {attempt} failed or interrupted: {e}")

			# Edge Case 2: Clean up partial download file on failure or CTRL+C
			if dest.exists():
				dest.unlink()

			if attempt == max_retries or isinstance(e, KeyboardInterrupt):
				raise


def ensure_dataset(
	config: DatasetConfig,
):
	print(f"\n=== Ensuring Dataset: {config.name} ===")

	# Edge Case 3: Configuration sanity checks before network requests
	if config.extract_zip:
		if config.extract_dir is None:
			raise ValueError(
				f"[{config.name}] `extract_dir` must be provided when `extract_zip=True`."
			)
		
		# Prevent collision where download_path matches check_file and gets deleted in finally block
		if config.download_path.resolve() == config.check_file.resolve():
			raise ValueError(
				f"[{config.name}] `download_path` and `check_file` cannot point to the exact same path "
				f"when `extract_zip=True`, because the temporary archive is unlinked after extraction."
			)

	# Check file check (returns False without raising exceptions if parent dirs don't exist)
	if config.check_file.exists():
		print(
			f"[{config.name}] Dataset already exists.\n"
			f"Location: {config.check_file}"
		)
		return

	print(f"[{config.name}] Dataset check failed (not found). Downloading...")

	try:
		download_with_retries(
			config.url,
			config.download_path,
		)

		if config.extract_zip:
			print(f"[{config.name}] Extracting archive...")

			config.extract_dir.mkdir(
				parents=True,
				exist_ok=True,
			)

			# Edge Case 4: Corrupt ZIP handling
			try:
				with zipfile.ZipFile(config.download_path, "r") as zip_ref:
					# Edge Case 5: Detect empty zip archives
					if not zip_ref.namelist():
						raise zipfile.BadZipFile("Downloaded ZIP file is empty.")
					
					zip_ref.extractall(config.extract_dir)

			except zipfile.BadZipFile as err:
				raise RuntimeError(
					f"[{config.name}] Downloaded file is corrupt or not a valid ZIP archive."
				) from err

			print(f"[{config.name}] Extraction complete.")

		# Edge Case 6: Post-extraction verification
		if not config.check_file.exists():
			raise FileNotFoundError(
				f"[{config.name}] Download/Extraction completed, but expected check_file "
				f"was not found at: {config.check_file}\n"
				f"Verify `check_file` relative path matches archive contents."
			)

		print(f"[{config.name}] Dataset ready at: {config.check_file}")

	finally:
		# Edge Case 7: Clean up zip file only if extract_zip=True and file exists
		if config.extract_zip and config.download_path.exists():
			config.download_path.unlink()
			print(f"[{config.name}] Temporary archive removed.")

In [7]:
GMTED2010 = DatasetConfig(
	name="GMTED2010",
	url=(
		"https://edcintl.cr.usgs.gov/downloads/"
		"sciweb1/shared/topo/downloads/"
		"GMTED/Grid_ZipFiles/mi30_grd.zip"
	),
	download_path=EXTERNAL_TOOLS_DIR / "mi30_grd.zip",
	extract_dir=EXTERNAL_TOOLS_DIR / "gmted2010",
	check_file=(
		EXTERNAL_TOOLS_DIR
		/ "gmted2010"
		/ "mi30_grd"
		/ "hdr.adf"
	),
	extract_zip=True,
)

VS30 = DatasetConfig(
	name="VS30",
	url="https://apps.usgs.gov/shakemap_geodata/vs30/global_vs30.grd",
	download_path=EXTERNAL_TOOLS_DIR /"global_vs30" /"global_vs30.grd",
	check_file=EXTERNAL_TOOLS_DIR / "global_vs30" / "global_vs30.grd",

	extract_zip=False,
)
ensure_dataset(GMTED2010)
ensure_dataset(VS30)


=== Ensuring Dataset: GMTED2010 ===
[GMTED2010] Dataset already exists.
Location: /home/keshav/Projects/AI_ML/SpatioTemporalEarthquakeProbabilisticPredictionSystem/src/spatiotemporal_earthquake_probabilistic_prediction_system/external_tools/gmted2010/mi30_grd/hdr.adf

=== Ensuring Dataset: VS30 ===
[VS30] Dataset already exists.
Location: /home/keshav/Projects/AI_ML/SpatioTemporalEarthquakeProbabilisticPredictionSystem/src/spatiotemporal_earthquake_probabilistic_prediction_system/external_tools/global_vs30/global_vs30.grd
